In [ ]:
from pathlib import Path
import json
import urllib.request
import numpy as np
import pandas as pd
from IPython.display import display

DATA_DIR = Path("../data")

WEI = 1e18
UNCORRELATED_BPS_BY_CHAIN = {
    "ethereum": 6.0, "gnosis": 4.0, "arbitrum": 1.0, "base": 3.0,
    "avalanche_c": 3.0, "polygon": 4.0, "bnb": 1.0,
}
CORRELATED_BPS = 0.3

TOKEN_LISTS_PATH = DATA_DIR / "token_lists.json"
TOKEN_LISTS_URL = "https://cms.cow.finance/api/correlated-tokens?pagination[pageSize]=100"

CHAIN_ALIASES = {
    "ethereum": ("mainnet", "ethereum"), "gnosis": ("gnosis", "xdai"),
    "arbitrum": ("arbitrum",), "base": ("base",), "polygon": ("polygon",),
    "bnb": ("bnb", "bsc"), "avalanche_c": ("avalanche",),
}


def load_inputs(data_dir):
    data_dir = Path(data_dir)
    prefixes = tuple(f"{chain}_" for chain in CHAIN_ALIASES)
    order_files = [
        path for path in sorted(data_dir.glob("*.csv"))
        if path.name.startswith(prefixes)
        and not path.name.endswith("_consistency_shares.csv")
    ]
    share_files = [
        path for path in sorted(data_dir.glob("*_consistency_shares.csv"))
        if path.name.startswith(prefixes)
    ]
    if not order_files:
        raise FileNotFoundError(f"No chain order CSVs found in {data_dir}")
    if not share_files:
        raise FileNotFoundError(f"No *_consistency_shares.csv files found in {data_dir}")
    print("Order files:")
    for path in order_files:
        print(" ", path.name)
    print("\nConsistency-share files:")
    for path in share_files:
        print(" ", path.name)
    orders = pd.concat(
        [pd.read_csv(path, low_memory=False) for path in order_files],
        ignore_index=True,
    ).drop_duplicates()
    shares = pd.concat(
        [pd.read_csv(path, low_memory=False) for path in share_files],
        ignore_index=True,
    ).drop_duplicates()
    return orders, shares


def load_correlated_groups(chains):
    if not TOKEN_LISTS_PATH.exists():
        TOKEN_LISTS_PATH.parent.mkdir(parents=True, exist_ok=True)
        with urllib.request.urlopen(TOKEN_LISTS_URL) as response:
            TOKEN_LISTS_PATH.write_bytes(response.read())

    payload = json.loads(TOKEN_LISTS_PATH.read_text())

    named_groups = [
        (
            entry["attributes"]["name"].lower(),
            {str(token).lower() for token in entry["attributes"]["tokens"]},
        )
        for entry in payload["data"]
    ]

    groups_by_chain = {}
    for chain in chains:
        aliases = CHAIN_ALIASES[chain]
        groups = [
            tokens for name, tokens in named_groups
            if any(alias in name for alias in aliases)
        ]
        if not groups:
            raise ValueError(f"No correlated-token groups found for {chain}")
        groups_by_chain[chain] = groups

    return groups_by_chain


def build_auction_solver(orders, groups_by_chain):
    required = {
        "blockchain", "auction_id", "solver", "solver_name",
        "accounting_period", "sell_token", "buy_token", "settled",
        "is_excluded_from_penalties", "volume_native", "reward_penalty_native",
        "reward_penalty_uncapped_native", "reward_cap_upper_native",
    }
    missing = sorted(required - set(orders.columns))
    if missing:
        raise KeyError(f"Order CSVs are missing required columns: {missing}")

    data = orders.copy()
    data["solver"] = data["solver"].astype(str).str.lower().str.strip()
    data["sell_token"] = data["sell_token"].astype(str).str.lower().str.strip()
    data["buy_token"] = data["buy_token"].astype(str).str.lower().str.strip()
    data["accounting_period"] = data["accounting_period"].astype(str)

    native_columns = [
        "volume_native", "reward_penalty_native",
        "reward_penalty_uncapped_native", "reward_cap_upper_native",
    ]
    for column in native_columns:
        data[column] = pd.to_numeric(data[column], errors="coerce") / WEI

    boolean_columns = ["settled", "is_excluded_from_penalties"]
    for column in boolean_columns:
        data[column] = (
            data[column].astype(str).str.lower()
            .map({"true": True, "false": False}).fillna(False)
        )

    data["correlated"] = False
    for chain, token_groups in groups_by_chain.items():
        chain_mask = data["blockchain"].eq(chain)
        sell_tokens = data.loc[chain_mask, "sell_token"]
        buy_tokens = data.loc[chain_mask, "buy_token"]
        correlated = np.zeros(chain_mask.sum(), dtype=bool)
        for token_group in token_groups:
            correlated |= (
                sell_tokens.isin(token_group).to_numpy()
                & buy_tokens.isin(token_group).to_numpy()
            )
        data.loc[chain_mask, "correlated"] = correlated

    failed_volume = np.where(~data["settled"], data["volume_native"].fillna(0), 0.0)
    data["failed_correlated_native"] = np.where(
        data["correlated"], failed_volume, 0.0,
    )
    data["failed_uncorrelated_native"] = np.where(
        ~data["correlated"], failed_volume, 0.0,
    )

    keys = ["blockchain", "auction_id", "solver"]
    volumes = data.groupby(keys, as_index=False).agg(
        failed_correlated_native=("failed_correlated_native", "sum"),
        failed_uncorrelated_native=("failed_uncorrelated_native", "sum"),
    )
    rewards = data.groupby(keys, as_index=False).agg(
        solver_name=("solver_name", "first"),
        accounting_period=("accounting_period", "first"),
        excluded=("is_excluded_from_penalties", "first"),
        current_batch_native=("reward_penalty_native", "first"),
        uncapped_batch_native=("reward_penalty_uncapped_native", "first"),
        upper_reward_cap_native=("reward_cap_upper_native", "first"),
    )
    return rewards.merge(volumes, on=keys, how="left", validate="one_to_one")


def build_scenarios(auction_solver):
    data = auction_solver.copy()

    configured_chains = set(UNCORRELATED_BPS_BY_CHAIN)
    loaded_chains = set(data["blockchain"].dropna().unique())
    missing_rates = sorted(loaded_chains - configured_chains)
    if missing_rates:
        raise ValueError(f"No uncorrelated-token rate configured for chains: {missing_rates}")

    data["performance_reward_native"] = data["current_batch_native"].clip(lower=0)
    data["uncapped_penalty_native"] = (-data["uncapped_batch_native"]).clip(lower=0)

    current = data.assign(
        scenario="current", uncorrelated_rate_bps=np.nan,
        correlated_rate_bps=np.nan, penalty_cap_native=np.nan,
        penalty_native=(-data["current_batch_native"]).clip(lower=0),
        net_batch_native=data["current_batch_native"],
    )

    uncorrelated_rate = data["blockchain"].map(UNCORRELATED_BPS_BY_CHAIN).astype(float)
    proposed_cap = (
        uncorrelated_rate / 1e4 * data["failed_uncorrelated_native"]
        + CORRELATED_BPS / 1e4 * data["failed_correlated_native"]
    )
    proposed_penalty = np.minimum(data["uncapped_penalty_native"], proposed_cap)
    proposed_penalty = np.where(data["excluded"], 0.0, proposed_penalty)

    proposed = data.assign(
        scenario="proposed", uncorrelated_rate_bps=uncorrelated_rate,
        correlated_rate_bps=CORRELATED_BPS, penalty_cap_native=proposed_cap,
        penalty_native=proposed_penalty,
        net_batch_native=data["performance_reward_native"] - proposed_penalty,
    )

    result = pd.concat([current, proposed], ignore_index=True)
    result["consistency_budget_native"] = (
        result["upper_reward_cap_native"] - result["net_batch_native"]
    )

    negative_budget = result[result["consistency_budget_native"] < -1e-9]
    if not negative_budget.empty:
        raise ValueError(
            "Negative consistency budget found:\n"
            + negative_budget[
                ["blockchain", "auction_id", "solver", "scenario",
                 "consistency_budget_native"]
            ].head(20).to_string(index=False)
        )

    return result


def prepare_consistency_shares(shares):
    shares = shares.copy()
    shares["solver"] = shares["solver"].astype(str).str.lower().str.strip()

    direct_share = pd.to_numeric(shares["consistency_reward_share"], errors="coerce")
    total_budget = pd.to_numeric(shares["total_consistency_budget"], errors="coerce")
    solver_reward = pd.to_numeric(shares["consistency_reward_native"], errors="coerce")
    calculated_share = solver_reward / total_budget.replace(0, np.nan)

    shares["consistency_reward_share"] = (
        direct_share.fillna(calculated_share).fillna(0)
    )

    return shares.groupby(
        ["blockchain", "accounting_period", "solver"], as_index=False,
    ).agg(consistency_reward_share=("consistency_reward_share", "first"))


def calculate_solver_payments(scenarios, shares):
    required = {"blockchain", "accounting_period", "solver", "consistency_reward_share"}
    missing = sorted(required - set(shares.columns))
    if missing:
        raise KeyError(f"Consistency CSVs are missing columns: {missing}")

    share_data = prepare_consistency_shares(shares)

    weekly_budget = scenarios.groupby(
        ["blockchain", "accounting_period", "scenario"], as_index=False,
    ).agg(consistency_budget_native=("consistency_budget_native", "sum"))

    share_sums = share_data.groupby(
        ["blockchain", "accounting_period"], as_index=False,
    ).agg(share_sum=("consistency_reward_share", "sum"))

    share_check = weekly_budget.merge(
        share_sums, on=["blockchain", "accounting_period"], how="left",
    )
    bad_periods = share_check[
        share_check["consistency_budget_native"].abs().gt(1e-9)
        & ~np.isclose(share_check["share_sum"].fillna(0), 1.0, atol=1e-8)
    ]
    if not bad_periods.empty:
        raise ValueError(
            "Consistency shares do not sum to one:\n"
            + bad_periods[
                ["blockchain", "accounting_period", "share_sum"]
            ].drop_duplicates().to_string(index=False)
        )

    allocated = share_data.merge(
        weekly_budget, on=["blockchain", "accounting_period"], how="inner",
    )
    allocated["consistency_reward_native"] = (
        allocated["consistency_reward_share"] * allocated["consistency_budget_native"]
    )

    weekly_batch = scenarios.groupby(
        ["blockchain", "accounting_period", "solver", "scenario"], as_index=False,
    ).agg(
        performance_reward_native=("performance_reward_native", "sum"),
        penalty_native=("penalty_native", "sum"),
        net_batch_native=("net_batch_native", "sum"),
    )

    payments = weekly_batch.merge(
        allocated[
            ["blockchain", "accounting_period", "solver", "scenario",
             "consistency_reward_native"]
        ],
        on=["blockchain", "accounting_period", "solver", "scenario"],
        how="outer",
    )

    numeric_columns = [
        "performance_reward_native", "penalty_native",
        "net_batch_native", "consistency_reward_native",
    ]
    payments[numeric_columns] = payments[numeric_columns].fillna(0)
    payments["total_payment_native"] = (
        payments["net_batch_native"] + payments["consistency_reward_native"]
    )

    solver_names = scenarios.groupby(
        ["blockchain", "solver"], as_index=False,
    ).agg(solver_name=("solver_name", "first"))

    result = payments.groupby(
        ["blockchain", "solver", "scenario"], as_index=False,
    ).agg(
        performance_reward_native=("performance_reward_native", "sum"),
        penalty_native=("penalty_native", "sum"),
        consistency_reward_native=("consistency_reward_native", "sum"),
        total_payment_native=("total_payment_native", "sum"),
    ).merge(solver_names, on=["blockchain", "solver"], how="left")

    current_payment = (
        result[result["scenario"].eq("current")]
        [["blockchain", "solver", "total_payment_native"]]
        .rename(columns={"total_payment_native": "current_total_native"})
    )

    result = result.merge(current_payment, on=["blockchain", "solver"], how="left")
    result["change_vs_current_native"] = (
        result["total_payment_native"] - result["current_total_native"].fillna(0)
    )
    result["solver_name"] = result["solver_name"].fillna(result["solver"].str[:10])

    totals = result.groupby(["blockchain", "scenario"])["total_payment_native"].sum().unstack()
    max_difference = totals.sub(totals["current"], axis=0).abs().max().max()
    assert max_difference < 1e-8, totals
    print("Total payments are unchanged across scenarios.")

    result["solver_name"] = result["solver_name"].fillna(result["solver"].str[:10])
    result = result[~result["solver_name"].astype(str).str.startswith("0x")].copy()

    return result


def run_counterfactual(data_dir=DATA_DIR):
    orders, shares = load_inputs(data_dir)

    chains = sorted(orders["blockchain"].dropna().unique())
    unknown_chains = sorted(set(chains) - set(CHAIN_ALIASES))
    if unknown_chains:
        raise ValueError(f"Missing CHAIN_ALIASES entries for: {unknown_chains}")

    correlated_groups = load_correlated_groups(chains)
    auction_solver = build_auction_solver(orders, correlated_groups)
    auction_scenarios = build_scenarios(auction_solver)
    solver_payments = calculate_solver_payments(auction_scenarios, shares)

    scenario_order = pd.CategoricalDtype(["current", "proposed"], ordered=True)
    solver_payments["scenario"] = solver_payments["scenario"].astype(scenario_order)

    for chain in chains:
        table = solver_payments[solver_payments["blockchain"].eq(chain)].copy()

        current_ranking = (
            table[table["scenario"].eq("current")]
            [["solver", "total_payment_native"]]
            .rename(columns={"total_payment_native": "_rank"})
        )

        table = (
            table.merge(current_ranking, on="solver", how="left")
            .sort_values(
                ["_rank", "solver_name", "scenario"],
                ascending=[False, True, True],
            )
            .drop(columns="_rank")
        )

        rate = UNCORRELATED_BPS_BY_CHAIN[chain]
        print(
            f"\n=== {chain} | uncorrelated={rate:g} bps, "
            f"correlated={CORRELATED_BPS:g} bps ==="
        )

        display(
            table[
                ["solver_name", "solver", "scenario", "performance_reward_native",
                 "penalty_native", "consistency_reward_native",
                 "total_payment_native", "change_vs_current_native"]
            ].round(6)
        )

    return solver_payments, auction_scenarios


SOLVER_PAYMENTS, AUCTION_SCENARIOS = run_counterfactual(DATA_DIR)